# OpenPlaque — Proximal-Trunk Topology Crosswalk v1

Compares the source-QC-positive long continuation against frozen LAD, C6/C7 structural paths, and RCA. No new vessel search and no frozen-master modification.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, time
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR=DRIVE_ROOT+'/Left_Proximal_Trunk_Topology_Crosswalk_v1'
REUSE_EXISTING_OUTPUTS=False
BRANCH='left-proximal-trunk-topology-crosswalk-from-main'
PINNED_SCIENCE_COMMIT='cd0844c4f60fcb1810ffccef7fa1b217ad318f67'
BASELINE='0593b453959f5a353d644267fbeef24b514ef4d7'
EXPECTED_ALGORITHM='left-proximal-trunk-topology-crosswalk-v1.0'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR,'notebook_started.json').write_text(json.dumps({'started':time.time(),'branch':BRANCH,'science_pin':PINNED_SCIENCE_COMMIT},indent=2))
print('Output:',OUTPUT_DIR)
print('Branch:',BRANCH)
print('Pinned science:',PINNED_SCIENCE_COMMIT)


In [ ]:
import os, shutil, subprocess
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
subprocess.run(['git','clone','--depth','20','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',repo],check=True)
subprocess.run(['git','-C',repo,'checkout','--detach',PINNED_SCIENCE_COMMIT],check=True)
head=subprocess.check_output(['git','-C',repo,'rev-parse','HEAD'],text=True).strip()
mb=subprocess.check_output(['git','-C',repo,'merge-base','HEAD',BASELINE],text=True).strip()
print('HEAD:',head)
print('Merge base:',mb)
assert head==PINNED_SCIENCE_COMMIT,(head,PINNED_SCIENCE_COMMIT)
assert mb==BASELINE,(mb,BASELINE)


In [ ]:
%pip uninstall -y openplaque >/dev/null 2>&1
%pip install -q --no-cache-dir --force-reinstall --no-deps /content/OpenPlaque
%pip install -q pytest SimpleITK scipy pandas matplotlib numpy


In [ ]:
import sys, importlib, pathlib, pytest
for name in list(sys.modules):
    if name=='openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
importlib.invalidate_caches()
import openplaque
from openplaque import left_proximal_trunk_topology_crosswalk_v1 as exp
print('openplaque:',openplaque.__file__)
print('experiment:',exp.__file__)
print('algorithm:',exp.ALGORITHM)
assert exp.BASELINE==BASELINE
assert exp.ALGORITHM==EXPECTED_ALGORITHM
compile(pathlib.Path(exp.__file__).read_text(),exp.__file__,'exec')
print('synthetic self-test:',exp.synthetic_topology_crosswalk_self_test())
rc=pytest.main(['-q','/content/OpenPlaque/tests/test_left_proximal_trunk_topology_crosswalk_v1.py'])
if rc!=0: raise RuntimeError(f'pytest failed: {rc}')


In [ ]:
from pathlib import Path
import json
root=Path(DRIVE_ROOT)
required=[
 root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 root/'Joint_Three_Vessel_Template_Classifier_v1/candidate_04_source_path.csv',
 root/'LCX_Distal_Reacquisition_v1_fixed/C7_extended_path.csv',
 root/'LCX_Structural_Identity_Adjudication_v1/structural_identity_decision.json',
 root/'Left_Proximal_Trunk_Local_Multidirection_v1/summary.json',
 root/'Left_Proximal_Trunk_Local_Multidirection_v1/best_local_multidirection_path.csv',
 root/'Left_Proximal_Trunk_Source_Led_Long_Extension_v1/summary.json',
 root/'Left_Proximal_Trunk_Source_Led_Long_Extension_v1/best_long_extension_path.csv',
 root/'Left_Proximal_Trunk_Continuation_QC_v1/accepted_proximal_trunk_continuation_candidate.csv',
]
missing=[str(p) for p in required if not p.exists()]
print('Preflight files:',len(required),'missing:',len(missing))
if missing: raise FileNotFoundError('\n'.join(missing))
local=json.loads(required[7].read_text())
longs=json.loads(required[9].read_text())
struct=json.loads(required[6].read_text())
print('Local status:',local.get('status'))
print('Long status:',longs.get('status'))
print('C6/C7 structural gates:',struct.get('all_predeclared_structural_gates_pass'))
Path(OUTPUT_DIR,'preflight_complete.json').write_text(json.dumps({'ok':True,'local_status':local.get('status'),'long_status':longs.get('status')},indent=2))


In [ ]:
import time
from openplaque.left_proximal_trunk_topology_crosswalk_v1 import run
t0=time.time()
result=run(DRIVE_ROOT,OUTPUT_DIR)
s=result['summary']
print('ELAPSED SEC:',round(time.time()-t0,2))
print('STATUS:',s.get('status'))
for k,v in s.get('associations',{}).items():
    print(k,'min=',round(v.get('min_distance_mm',float('nan')),3),'span<=2=',round(v.get('max_contiguous_span_within_2mm',0),2),'align=',round(v.get('median_tangent_alignment_within_2mm',0),3),'PASS=',v.get('association_gate_pass'))
print('REPORT:',result.get('report'))
print('ZIP:',result.get('zip'))
